# Bayesian Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChemAI-Lab/AI4Chem/blob/main/website/modules/06-bayesian_optimization.ipynb)

**References:**
1. **Chapters 1-3**: [Bayesian Optimization](https://bayesoptbook.com/book/bayesoptbook.pdf), R. Garnett
2. **Chapters 6**: [Pattern Recognition and Machine Learning](https://www.microsoft.com/en-us/research/wp-content/uploads/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf), C. M. Bishop.
3. **Chapter 2**:  [Gaussian Processes for Machine Learning](https://direct.mit.edu/books/oa-monograph-pdf/2514321/book_9780262256834.pdf), C. E. Rasmussen, C. K. I. Williams
4. **Chapter 4**: [Machine Learning in Quantum Sciences](https://arxiv.org/pdf/2204.04198)
5. **Chapter 6**: [Probabilistic Machine Learning: An Introduction, K. P. Murphy.](https://probml.github.io/pml-book/book1.html)
6. [**The Kernel Cookbook**](https://www.cs.toronto.edu/~duvenaud/cookbook/)

**Power Point Slides**: [![Bayesian Optimization Slides](https://img.shields.io/badge/Slides–Download-PPTX-success?logo=microsoftpowerpoint&logoColor=white)](https://raw.githubusercontent.com/ChemAI-Lab/AI4Chem/main/website/modules/BayesOpt.pptx)



# Optimization without Gradients

During the course, we saw that many scientific problems can be recast as an optimization problem,
$$
\mathbf{x}^* = \arg\min f(\mathbf{x})
$$
where $\mathbf{x}^*$ is the **minimizer** of the function $f$. <br>
We have solved this problem using gradient-based methods, where at each step we used the local information of the gradient to move "towards" the minimizer of $f$, using
$$
\mathbf{x}^*_{t+1} = \mathbf{x}^*_{t} - \eta \nabla_{\mathbf{x}}f.
$$
These style of methods have been successful in scientific frameworks where $f$ is differentiable, or its gradient can be easily estimated. However, for other systems where $f$ is a **black box** function, gradient-based optimization is unfeasible.

## Black Box Function
* A black box function is a system, algorithm, or piece of code where only the inputs and outputs are visible, while the internal logic, mechanisms, or code structure are hidden or unknown.

```
x ∈ ℝ^d   ─────▶   [   BLACK BOX  f(x)   ]   ─────▶   y = f(x)
(parameters)                                   (objective value)
```

*  Expensive (minutes–days)
*  No gradients available  
*  Noisy observations  <br>


> In **Bayesian Optimization**, we assume we can query the function, but we cannot inspect its internals. <br>
> We cannot differentiate it analytically, and each evaluation may cost minutes, hours, or even days. <br>
> Therefore, we must be strategic about where to sample next.


Any Bayesian Optimization algorithm is composed of three main ingredients,
1. **Surrogate models** --> Approximates $f(\mathbf{x}$)
2. **Acquisition functions** --> Quantifies the information gain if a new point is known
3. **Exploration vs exploitation** --> Uncertainty vs Certainty

In [ ]:
!pip install py3Dmol
!pip install rdkit
!pip install pyscf
!pip install botorch

In [ ]:
from botorch.acquisition import ExpectedImprovement, UpperConfidenceBound, LogExpectedImprovement
from gpytorch.settings import fast_pred_var
from gpytorch.mlls import ExactMarginalLogLikelihood
# from gpytorch.models import gp
from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.models.transforms import Normalize
from botorch.optim import optimize_acqf

import torch
import numpy as np
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt


# import py3Dmol
# import rdkit
# import pyscf
# from pyscf import dft
# from rdkit import Chem
# from rdkit.Chem import Draw, rdDetermineBonds, MolFromXYZBlock
# from rdkit.Chem import rdDetermineBonds
# from rdkit.Chem.Draw import IPythonConsole
# IPythonConsole.ipython_3d = True



In [ ]:
def objective(x):
	return np.sin(x) + np.sin((10.0 / 3.0) * x)

# define optimal input value
x_optima = 5.145735
# hola

## Surrogate model

Since the evaluations of $f$ are expensive, we want to make as few function evaluations as possible.
This suggest the need to approximate $f$ with a model, which in the Bayesian optimization literature is known as **surrogate model**, $f_{\mathbf{\theta}}$, and will be trained with the collected data.


In [ ]:
x = np.random.uniform(0, 10, size=(4, 1))
y = objective(x)

x_grid = np.linspace(0, 10, 100).reshape(-1, 1)
y_grid = objective(x_grid)
x_grid_torch = torch.tensor(x_grid).float()

In [ ]:
train_x = torch.tensor(x).float()
train_y = torch.tensor(y).float()
print(train_x.shape, train_y.shape)

gp_model = SingleTaskGP(
    train_x,
    train_y,
    input_transform=Normalize(d=train_x.shape[-1]),
)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)

In [ ]:
from gpytorch.settings import fast_pred_var
gp_model.eval()
with torch.no_grad(), fast_pred_var():
    posterior = gp_model.posterior(x_grid_torch)          # BoTorch Posterior object
    mean = posterior.mean                        # shape: (n_test, m)  (m=1 usually)
    var  = posterior.variance                    # shape: (n_test, m)
    std  = var.sqrt()

plt.plot(x_grid, y_grid,c='k',label='Objective Function')
plt.scatter(x, y, color='k',s=55, marker='x', label='Samples')

mu = mean.squeeze().cpu().numpy()
sigma = std.squeeze().cpu().numpy()

# plot GP
plt.axvline(x=x_optima, ls='--', color='red',label = "Optima")
plt.plot(x_grid, mu, linewidth=2, label="GP mean")
plt.fill_between(
    x_grid.ravel(),
    mu - 2*sigma,
    mu + 2*sigma,
    alpha=0.2,
    label="±2σ"
)
plt.xlabel("x",fontsize=16)
plt.ylabel(f"$f(x)$",fontsize=16)
plt.legend()

## Acquisition Function $\alpha(x)$

Acq. Function evaluates the expected utility of each possible point we can querry.

### Upper Confidence Bounds (UCB)
  $$
  \alpha_{UCB}(x) = \mu(x) + \beta \sigma(x)
  $$
  Interpretation:
  * Pick points that either have high predicted value or high uncertainty.

### Expected Improvement (EI)
  $$
  I(x) = \max(0,f(x) - f^*) \\
  \alpha_{EI}(x) = (\mu(x) -f^*)\Phi(Z) + \sigma(x)\phi(Z)
  $$
  where
  * $Z = (\mu(x) -f^*?)/\sigma(x)$
  * $\Phi$: normal CDF
  * $\phi$: normal PDF

  Interpretation:
  * First term → exploitation
  * Second term → exploration
  * Automatically balances both

### Log Expected Improvement (LogEI)

  More numerically stable version:
  $$
  \alpha_{LogEI}(x)=\log⁡(\alpha_{EI}(x))
  $$

Useful when EI becomes very small.

In [ ]:
# --- GP posterior ---
gp_model.eval()
with torch.no_grad(), fast_pred_var():
    posterior = gp_model.posterior(x_grid_torch)
    mean = posterior.mean
    std = posterior.variance.sqrt()

mu = mean.squeeze().cpu().numpy()
sigma = std.squeeze().cpu().numpy()

xg = x_grid.squeeze()  # (n,) for plotting

# --- Acquisition on the same grid ---
best_f = torch.max(torch.tensor(y, dtype=x_grid_torch.dtype, device=x_grid_torch.device))
acq_fun_ei = ExpectedImprovement(model=gp_model, best_f=best_f, maximize=False)
acq_fun_ucb = UpperConfidenceBound(model=gp_model, beta=1., maximize=False)

with torch.no_grad():
    acq_vals_ei = acq_fun_ei(x_grid_torch.unsqueeze(1) if x_grid_torch.ndim == 2 else x_grid_torch)
    acq_vals_ei = acq_vals_ei.squeeze().cpu().numpy()
    acq_vals_ucb = acq_fun_ucb(x_grid_torch.unsqueeze(1) if x_grid_torch.ndim == 2 else x_grid_torch)
    acq_vals_ucb = acq_vals_ucb.squeeze().cpu().numpy()

# --- Next best point (maximize acquisition) ---
idx_next = acq_vals_ei.argmax()
x_ei_next = xg[idx_next]
acq_ei_next = acq_vals_ei[idx_next]

idx_next = acq_vals_ucb.argmax()
x_ucb_next = xg[idx_next]
acq_ucb_next = acq_vals_ucb[idx_next]

# --- Figure with stacked panels ---
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(9, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.05}
)

# ===== Top panel: objective + GP =====
ax1.plot(xg, y_grid, c="k", label="Objective Function")
ax1.scatter(x.squeeze(), y.squeeze(), color="k", s=55, marker="x", label="Samples")

ax1.axvline(x=x_optima, ls="--", color="red", label="Optima")
ax1.plot(xg, mu, linewidth=2, label="GP mean")

ax1.fill_between(
    xg,
    mu - 2 * sigma,
    mu + 2 * sigma,
    alpha=0.2,
    label="±2σ"
)

ax1.set_ylabel(r"$f(x)$", fontsize=16)
ax1.legend(loc="best")

# ===== Bottom panel: acquisition =====
ax2.plot(xg, acq_vals_ei, linewidth=2, color="tab:orange",label=r"$\alpha_{EI}$")
ax2.plot(xg, acq_vals_ucb, linewidth=2, color="tab:green",label=r"$\alpha_{UCB}$")

ax2.axvline(x=x_ei_next, ls=":", color="black", label=r"$x_{EI}$")
ax2.axvline(x=x_ucb_next, ls="--", color="black", label=r"$x_{UCB}$")

ax2.set_xlabel("x", fontsize=16)
ax2.set_ylabel(r"$\alpha(x)$", fontsize=14)
ax2.legend(loc="best")

plt.tight_layout()
plt.show()
#

# Bayesian Optimization

<img src="https://github.com/ChemAI-Lab/AI4Chem/raw/main/website/modules/Figures/BO_Algo.png" width="500" alt="Chem AI Logo">



In [ ]:
import warnings
warnings.filterwarnings("ignore")

xg = np.asarray(x_grid).reshape(-1)        # (N,)
yg = np.asarray(y_grid).reshape(-1)        # (N,)
x_obs = np.asarray(x[:1]).reshape(-1)          # (1,)
y_obs = np.asarray(y[:1]).reshape(-1)          # (1,)

# Optional: "true optimum" on the grid (for visual reference)
x_opt = xg[np.argmin(yg)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double

# Grid tensor for model queries
X_grid_t = torch.tensor(xg, device=device, dtype=dtype).unsqueeze(-1)  # (N,1)

# Fit GP with new data
def fit_gp(X_np, Y_np):
    """Fit a SingleTaskGP on 1D data (X: (n,), Y: (n,))"""
    X_t = torch.tensor(X_np, device=device, dtype=dtype).unsqueeze(-1)  # (n,1)
    Y_t = torch.tensor(Y_np, device=device, dtype=dtype).unsqueeze(-1)  # (n,1)
    model = SingleTaskGP(X_t, Y_t, outcome_transform=Standardize(m=1))
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)
    return model

# ---- animation target: total samples = 10 ----
n_target = 20
n0 = len(x_obs)
n_steps = max(0, n_target - n0)

# Track history for animation frames
history = [(x_obs.copy(), y_obs.copy())]

# BO loop (precompute points so animation is deterministic & fast)
for _ in range(n_steps):
    model = fit_gp(x_obs, y_obs)
    model.eval()

    # best_f should be min observed for minimize=True (maximize=False)
    # best_f = torch.tensor(np.min(y_obs), device=device, dtype=dtype)
    # acq = ExpectedImprovement(model=model, best_f=best_f, maximize=False)
    acq = UpperConfidenceBound(model=model, beta=0.2, maximize=False)

    with torch.no_grad():
        # Corrected: Add unsqueeze(1) to make X_grid_t shape (N, 1, 1)
        acq_vals = acq(X_grid_t.unsqueeze(1)).squeeze(-1).cpu().numpy()  # (N,)

    # Next point = argmax of acquisition (ALWAYS maximize acquisition)
    idx_next = int(np.argmax(acq_vals))
    x_next = xg[idx_next]

    # Evaluate objective at x_next:
    y_next = np.interp(x_next, xg, yg)


    taken = set(np.round(x_obs, 12))
    if np.round(x_next, 12) in taken:
        order = np.argsort(-acq_vals)
        for j in order:
            cand = xg[int(j)]
            if np.round(cand, 12) not in taken:
                x_next = cand
                y_next = np.interp(x_next, xg, yg)
                break

    # Append
    x_obs = np.append(x_obs, x_next)
    y_obs = np.append(y_obs, y_next)

    history.append((x_obs.copy(), y_obs.copy()))

# -----------------------------
# Build animation
# -----------------------------
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(9, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
)

line_mu, = ax1.plot([], [], lw=2, label="GP mean")
band = None
scat = ax1.scatter([], [], s=55, marker="x", color="k", label="Samples")
ax1.plot(xg, yg, c="k", lw=1.5, label="Objective Function")
ax1.axvline(x=x_opt, ls="--", color="red", label="True optimum (grid)")

line_acq, = ax2.plot([], [], lw=2, label=r"$\alpha$ (minimize)")
next_pt = ax2.scatter([], [], s=80, marker="o", zorder=3, label="Next query")
vline_next = ax2.axvline(x=xg[0], ls=":", color="black")

ax1.set_ylabel(r"$f(x)$", fontsize=16)
ax2.set_ylabel(r"$\alpha(x)$", fontsize=14)
ax2.set_xlabel("x", fontsize=16)

ax1.legend(loc="best")
ax2.legend(loc="best")

def init():
    line_mu.set_data([], [])
    line_acq.set_data([], [])
    next_pt.set_offsets(np.empty((0, 2)))
    return (line_mu, line_acq, next_pt, vline_next)

def update(frame):
    global band
    X_np, Y_np = history[frame]
    model = fit_gp(X_np, Y_np)
    model.eval()

    # GP posterior
    with torch.no_grad(), fast_pred_var():
        post = model.posterior(X_grid_t)
        mu = post.mean.squeeze(-1).cpu().numpy()
        sigma = post.variance.sqrt().squeeze(-1).cpu().numpy()

    # Acquisition
    best_f = torch.tensor(np.min(Y_np), device=device, dtype=dtype)
    acq = ExpectedImprovement(model=model, best_f=best_f, maximize=False)
    with torch.no_grad():
        # Corrected: Add unsqueeze(1) to make X_grid_t shape (N, 1, 1)
        acq_vals = acq(X_grid_t.unsqueeze(1)).squeeze(-1).cpu().numpy()

    idx_next = int(np.argmax(acq_vals))
    x_next = xg[idx_next]
    a_next = acq_vals[idx_next]

    # Update top panel: samples + GP
    scat.set_offsets(np.c_[X_np, Y_np])
    line_mu.set_data(xg, mu)

    # Update uncertainty band (recreate each frame)
    if band is not None:
        band.remove()
    band = ax1.fill_between(xg, mu - 2*sigma, mu + 2*sigma, alpha=0.2, label="±2σ",color="tab:blue")

    # Update bottom panel: acquisition + next point
    line_acq.set_data(xg, acq_vals)
    next_pt.set_offsets(np.array([[x_next, a_next]]))
    vline_next.set_xdata([x_next, x_next])

    ax1.set_title(f"Bayesian Optimization (frame {frame+1}/{len(history)} | samples={len(X_np)})")
    return (scat, line_mu, line_acq, next_pt, vline_next, band)

anim = FuncAnimation(fig, update, frames=len(history), init_func=init,
                     interval=800, blit=False, repeat=False)

# plt.show()

# In Jupyter, display the animation like this:
from IPython.display import HTML
plt.close(fig)   # prevents static duplicate rendering
HTML(anim.to_jshtml())

## Realistic Application

Generate a Potential Energy Surface for the water molecule

In [ ]:
import py3Dmol
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, rdDetermineBonds, MolFromXYZBlock
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_3d = True

In [ ]:
def draw_with_spheres(xyz):
    raw_mol = Chem.MolFromXYZBlock(xyz)
    conn_mol = Chem.Mol(raw_mol)
    rdDetermineBonds.DetermineConnectivity(conn_mol)

    v = py3Dmol.view(width=400,height=400)
    IPythonConsole.addMolToView(conn_mol,v)
    v.zoomTo()
    v.setStyle({'sphere':{'radius':0.35},'stick':{'radius':0.1}});
    v.show()

In [ ]:
xyz = '''3
* (null), Energy   -1000.0000000
H     0.00000     0.7554     -0.47116
H     0.00000    -0.75545     -0.4711
O     0.00000     0.00000     0.11779
'''

draw_with_spheres(xyz)

In [ ]:
import numpy as np
import pyscf
from pyscf import dft

def get_xyz_matrix(angle, dist, n_atoms = 3):
    xyz = []

    mol = pyscf.gto.Mole()
    mol.atom = '''
      O
      H  1  	1.2
      H  1  %.3f  2 %.3f
    '''%(dist,angle)
    mol.unit = 'Angstrom'
    mol.build()
    for i in range(n_atoms):
        xyzi = mol.atom_coord(i).tolist()
        xyzi = [mol.atom_symbol(i)] + xyzi
        xyz.append(xyzi)

    xyz_str = '%s\n Generated by PySCF\n'%(n_atoms)
    for xyzi in xyz:
        print(xyzi)
        xyzi_str = '%s     %.4f     %.4f     %.4f\n'%(xyzi[0],xyzi[1],xyzi[2],xyzi[3])
        xyz_str += xyzi_str
    return xyz_str

In [ ]:
def energy_water(angle,dist):
    mol = pyscf.gto.Mole()
    mol.atom = '''
      O
      H  1  	1.2
      H  1  %.3f  2 %.3f
    '''%(dist,angle)
    mol.unit = 'Angstrom'
    mol.basis = 'sto-3g' # basis set level
    mol.build()
    rks_h2o = dft.RKS(mol)
    rks_h2o.xc = 'b3lyp' # XC functional
    energy =rks_h2o.kernel()

    xyz = f'3\nGenerated by PySCF\n' #+ xyz
    return energy, xyz

In [ ]:
# angle_list = np.linspace(-5., 5., 10)  + 111.413 #  111.413 in radians is 1.94452367952 	#np.linspace(-0.25, 0.25, 5)
n_grid = 12
angle_list = np.linspace(40., 160., n_grid)
dist_list = np.linspace(0.75, 1.7, n_grid)
print("Angles: ", angle_list)
print("R_OH: ", dist_list)

In [ ]:
# generate Data for plotting
xyz_all = []
pes = []

X,Y = np.meshgrid(angle_list,dist_list)
for angle,dist in zip(X.flatten(),Y.flatten()):
      # pes_i, xyz_i = scan_pes(angle,dist)
      pes_i, xyz_i = energy_water(angle,dist)
      pes.append(pes_i)
      xyz_all.append(xyz_i)

In [ ]:
# ground state geometry

i_min = np.argmin(pes)
print(i_min)
angles_grid = X.flatten()
dist_grid = Y.flatten()
print('Ground state geometry')
print(angles_grid[i_min],dist_grid[i_min])
print('Energy', pes[i_min])

x_min = np.array([angles_grid[i_min],dist_grid[i_min]])
e_min = pes[i_min]

In [ ]:
# PES Plot

D = {'Energy':np.asarray(pes),
     'X':np.column_stack((X.flatten(),Y.flatten()))}

# X,Y = np.meshgrid(angle_list,dist_list)
Z = np.asarray(pes).reshape(X.shape)

plt.figure(figsize=(10,10))
plt.contourf(X,Y,Z,levels=10)
plt.scatter(x_min[0],x_min[1],marker='x',c='w',s=75)
plt.xlabel('H-O-H Angle',fontsize=15)
plt.ylabel('H-O Bond distance',fontsize=15)

In [ ]:
# define a GP for Bayesian Optimization

# random initial point
i0 = np.random.randint(len(pes))
X_train = torch.tensor(np.array([[angles_grid[i0],dist_grid[i0]]])).float()
y_train = torch.tensor(pes[i0].reshape(-1,1)).float()
y_best = y_train[0]

bounds = torch.tensor([[50.,0.65],[160,2.]])


n_itr = 25
candidates = []

X_train_bo = X_train.clone()
y_train_bo = y_train.clone()
y_energy_bo = []

for i in range(n_itr):

    # Fit GP on current data (raw y); Standardize happens internally
    gp_model = SingleTaskGP(
        X_train_bo, y_train_bo,
        outcome_transform=Standardize(m=1)
    )
    mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
    fit_gpytorch_mll(mll)

    # Acquisition (still set maximize according to your real objective)
    UCB = UpperConfidenceBound(gp_model, beta=0.1, maximize=False)

    candidate, acq_value = optimize_acqf(
        UCB, bounds=bounds, q=1, num_restarts=5, raw_samples=20
    )

    candidates.append(candidate.detach().cpu().numpy()[0])

    # Evaluate your true objective at the candidate
    xi = candidate.detach().cpu().numpy()[0]
    angle, dist = float(xi[0]), float(xi[1])

    y_energy0, _ = energy_water(angle, dist)   # raw energy (not standardized)
    y_energy_bo.append(y_energy0)

    y_new = y_energy0

    if y_energy0 < y_best:
      y_best = y_energy0
      print("New GS Geom found", candidate, y_best)


    # Append new observation
    X_train_bo = torch.vstack((X_train_bo, candidate))
    y_train_bo = torch.vstack((y_train_bo, torch.tensor([[y_new]], dtype=torch.float32)))

    print(i, xi, y_energy0)

y_energy_bo = np.array(y_energy_bo)

In [ ]:
plt.plot(np.arange(y_energy_bo.shape[0]),y_energy_bo,marker='x', label='BO Samples')
plt.hlines(e_min, 0,y_train_bo.detach().shape[0], color='k',ls = '--', label = "Grid Search Best")
plt.xlabel('Iterations')
plt.ylabel('Energy of the candidate point')
plt.legend()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

candidates = np.asarray(candidates)

# Precompute arrays once (avoid repeated .detach() in animation)
X_seen = X_train_bo.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(5, 5))

# Draw static background once
cf = ax.contourf(X, Y, Z, levels=10)
ax.set_xlabel("H-O-H Angle", fontsize=15)
ax.set_ylabel("H-O Bond distance", fontsize=15)

# Static marker for true minimum
min_sc = ax.scatter(x_min[0], x_min[1], marker="x", c="w", s=75)

# Artists we will update
past_sc = ax.scatter([], [], color="w", s=30, marker="s")
curr_sc = ax.scatter([], [], color="r", s=30, marker="o")
title = ax.set_title("")

def init():
    past_sc.set_offsets(np.empty((0, 2)))
    curr_sc.set_offsets(np.empty((0, 2)))
    title.set_text("")
    return past_sc, curr_sc, title

def update(i):
    # Past BO points up to i (use candidates, or use X_seen if you prefer)
    # Using candidates keeps it aligned with "next points"
    if i > 0:
        past_sc.set_offsets(candidates[:i, :2])
    else:
        past_sc.set_offsets(np.empty((0, 2)))

    # Current candidate
    curr_sc.set_offsets(candidates[i, :2])

    title.set_text(f"BO step {i+1}/{len(candidates)}")
    return past_sc, curr_sc, title

anim = FuncAnimation(
    fig, update, frames=len(candidates),
    init_func=init, interval=600, blit=True, repeat=False
)

plt.close(fig)  # prevents duplicate static plot in notebook
HTML(anim.to_jshtml())
